In [ ]:
# 1. Install all necessary libraries
!pip install -q openai-whisper moviepy sentence-transformers transformers Pillow opencv-python jellyfish

# 2. Install the system-level tool for media processing
!apt-get install -qq ffmpeg

!pip install -q gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 21.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.5/360.5 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 4.6 MB/s eta 0:00:00


In [ ]:
import whisper
import cv2
import os
import jellyfish
import torch
from moviepy.editor import VideoFileClip
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
from sentence_transformers import SentenceTransformer, util

# --- 1. CORE PROCESSING FUNCTIONS ---

def process_video_to_text(video_path):
    # A. Visual Branch: Multi-Scene Sampling
    print("👁️ Analyzing multiple scenes for visual context...")
    processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
    model_v = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

    vidcap = cv2.VideoCapture(video_path)
    fps = vidcap.get(cv2.CAP_PROP_FPS)
    total_frames = int(vidcap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Define 3 timestamps: 10%, 50%, and 90% through the video
    sample_points = [int(total_frames * 0.1), int(total_frames * 0.5), int(total_frames * 0.9)]
    scene_descriptions = []

    for frame_idx in sample_points:
        vidcap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        success, image = vidcap.read()
        if success:
            raw_image = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
            inputs = processor(raw_image, return_tensors="pt")
            out = model_v.generate(**inputs)
            desc = processor.decode(out[0], skip_special_tokens=True)
            if desc not in scene_descriptions: # Avoid duplicates if scene hasn't changed
                scene_descriptions.append(desc)

    # Merge all unique scenes into one visual summary
    visual_desc = ". ".join(scene_descriptions)

    # B. Audio Branch: MoviePy + Whisper (Remains the same)
    print("🔊 Transcribing full audio...")
    video = VideoFileClip(video_path)
    audio_path = "temp_audio.mp3"
    video.audio.write_audiofile(audio_path, logger=None)
    model_w = whisper.load_model("medium")
    transcript = model_w.transcribe(audio_path)['text'].strip()

    return transcript, visual_desc

def calculate_multimodal_score(student_text, visual_text, reference_text, target_keywords):
    """Calculates Semantic Similarity and Keyword Accuracy."""
    # SBERT for deep semantic meaning
    model_s = SentenceTransformer('all-MiniLM-L6-v2')
    student_context = f"Speech: {student_text}. Visual Context: {visual_text}."

    emb1 = model_s.encode(student_context, convert_to_tensor=True)
    emb2 = model_s.encode(reference_text, convert_to_tensor=True)
    semantic_score = round(util.cos_sim(emb1, emb2).item() * 100, 2)

    # Jellyfish for technical string audit
    found_keywords = []
    missing_keywords = []
    student_words = student_text.lower().split()

    for kw in target_keywords:
        match_found = any(jellyfish.levenshtein_distance(kw.lower(), word) <= 1 for word in student_words)
        if match_found:
            found_keywords.append(kw)
        else:
            missing_keywords.append(kw)

    keyword_pct = round((len(found_keywords) / len(target_keywords)) * 100, 2)

    return semantic_score, keyword_pct, found_keywords, missing_keywords

# --- 2. PIPELINE WRAPPER ---

def run_evaluation_pipeline(video_file, ref_ans, keywords):
    transcript, visuals = process_video_to_text(video_file)
    similarity, key_match, found, missing = calculate_multimodal_score(
        transcript, visuals, ref_ans, keywords
    )

    return {
        "transcript": transcript,
        "visual_context": visuals,
        "semantic_similarity": similarity,
        "keyword_accuracy": key_match,
        "found_terms": found,
        "missing_terms": missing,
        "word_count": len(transcript.split())
    }

# --- 3. TEST EXECUTION ---
# Ensure your video file is uploaded to the sidebar!
INPUT_VIDEO = "/content/Girl.mp4"
REFERENCE = "The speaker describes the first time she realized she was famous. She shares an anecdote about visiting a grocery store and being surprised by the presence of many photographers, initially not realizing they were there for her."
KEYWORDS_LIST = ["famous","realized","whole","foods","surprised"]

if os.path.exists(INPUT_VIDEO):
    final_output = run_evaluation_pipeline(INPUT_VIDEO, REFERENCE, KEYWORDS_LIST)

    print("\n" + "⭐" * 20 + " EVALUATION REPORT " + "⭐" * 20)
    print(f"Transcript: {final_output['transcript']}")
    print(f"Visual Detection: {final_output['visual_context']}")
    print("-" * 50)
    print(f"Semantic Match: {final_output['semantic_similarity']}%")
    print(f"Technical Keyword Match: {final_output['keyword_accuracy']}%")
    print(f"Missing Key Terms: {final_output['missing_terms']}")
    print("=" * 60)
else:
    print(f"❌ Error: {INPUT_VIDEO} not found. Please upload it to the Colab sidebar.")


/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



❌ Error: /content/Girl.mp4 not found. Please upload it to the Colab sidebar.


In [ ]:
import gradio as gr

def gradio_interface(video_file, reference_ans, keywords_input):
    """
    Connects the Gradio UI to your backend pipeline.
    """
    # 1. Clean up the keywords (convert comma-separated string to a list)
    keyword_list = [k.strip() for k in keywords_input.split(",")]

    # 2. Run your existing backend function
    # Note: Using your established run_evaluation_pipeline from the notebook
    results = run_evaluation_pipeline(video_file, reference_ans, keyword_list)

    # 3. Return all 5 values for the UI
    return (
        results['transcript'],         # Audio Transcription
        results['visual_context'],      # Visual Context
        results['semantic_similarity'], # Semantic Match
        results['keyword_accuracy'],    # Keyword Match
        ", ".join(results['missing_terms']) if results['missing_terms'] else "None" # Missing
    )

# --- Define the Gradio UI Layout ---
with gr.Blocks(title="ViText: Multimodal Evaluation System") as demo:
    gr.Markdown("# 🎥 ViText: Multimodal Video Answer Evaluation")
    gr.Markdown("Upload a student video and compare it against an expert reference answer using AI.")

    with gr.Row():
        with gr.Column():
            # Input Section
            input_video = gr.Video(label="Step 1: Upload Student Video")
            input_ref = gr.Textbox(label="Step 2: Expert Reference Answer", lines=3)
            input_keys = gr.Textbox(label="Step 3: Technical Keywords (comma separated)", placeholder="e.g. photosynthesis, chlorophyll, sunlight")
            submit_btn = gr.Button("Evaluate Answer", variant="primary")

        with gr.Column():
            # Output Section
            out_audio = gr.Textbox(label="🔊 Audio Transcription")
            out_visual = gr.Textbox(label="👁️ Visual Context (Detected Scenes)")

            with gr.Row():
                out_semantic = gr.Number(label="Semantic Similarity (%)")
                out_keyword = gr.Number(label="Keyword Match (%)")

            out_missing = gr.Textbox(label="❌ Missing Technical Keywords")

    # Connect the button to the backend logic
    submit_btn.click(
        fn=gradio_interface,
        inputs=[input_video, input_ref, input_keys],
        outputs=[out_audio, out_visual, out_semantic, out_keyword, out_missing]
    )

# Launch the app with a public shareable link for your demo
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7629d4b4643c8a2e8c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
